Imports and Path Setup: This cell handles the environment and ensures your custom modules are visible.

In [2]:
import sys
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# --- 1. DYNAMIC PATH SETUP ---
CURRENT_FILE = Path(os.getcwd()) # In notebooks, use getcwd
PROJECT_ROOT = CURRENT_FILE
while PROJECT_ROOT.name != 'MRI_Based_Tumor_Detection-main' and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import Custom Modules
from src.models.cnn_backbone import tumorCNN
from src.data.dataset import MRIDataset
from src.preprocessing.transforms import get_train_transforms, get_val_transforms
from src.utils.seed import seed_everything

print(f"Project Root: {PROJECT_ROOT}")

Project Root: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main


Configuration and Data Loaders:  This cell defines where your images are and prepares the data streams.

In [3]:
seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training will run on: {device}")

# Define Paths
train_path = PROJECT_ROOT / "data" / "raw" / "train"
val_path = PROJECT_ROOT / "data" / "raw" / "val"
checkpoint_dir = PROJECT_ROOT / "experiments" / "models"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Dataset Initialization
train_ds = MRIDataset(root_dir=str(train_path), transform=get_train_transforms(image_size=128))
val_ds = MRIDataset(root_dir=str(val_path), transform=get_val_transforms(image_size=128))

# Data Loaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"Ready to train on {len(train_ds)} images.")
print(f"Validation set contains {len(val_ds)} images.")

Training will run on: cpu
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_0853.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_0943.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_1364.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_1371.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_1386.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\train\pituitary\Tr-pi_1401.jpg
Skipping corrupted image: c:\Users\zdmes\Documents\uni\DL LAB\MRI_Based_Tumor_Detection-main\data\raw\val\pituitary\Tr-pi_0857.jpg
Skipping corrupted image: c:\Users\zdmes\Docu

Model, Optimizer, and Training Loop: This cell executes the actual training.

In [4]:
# --- MODEL SETUP ---
model = tumorCNN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- TRAINING LOOP ---
epochs = 50
best_val_loss = float('inf')

for epoch in range(epochs):
    # TRAINING PHASE
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # VALIDATION PHASE
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    # Metrics and Logging
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = 100. * train_correct / train_total
    val_acc = 100. * val_correct / val_total
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_path = checkpoint_dir / 'best_cnn_v1.pth'
        torch.save({'model_state_dict': model.state_dict(), 'val_acc': val_acc}, str(save_path))
        print(f"Saved New Best Model (Acc: {val_acc:.2f}%)")

Epoch 01/50 | Train Loss: 0.8704 | Train Acc: 65.25% | Val Loss: 0.7820 | Val Acc: 68.63%
Saved New Best Model (Acc: 68.63%)
Epoch 02/50 | Train Loss: 0.6698 | Train Acc: 72.91% | Val Loss: 0.5474 | Val Acc: 80.85%
Saved New Best Model (Acc: 80.85%)
Epoch 03/50 | Train Loss: 0.6005 | Train Acc: 76.78% | Val Loss: 0.6700 | Val Acc: 73.34%
Epoch 04/50 | Train Loss: 0.5800 | Train Acc: 77.94% | Val Loss: 0.7244 | Val Acc: 72.61%
Epoch 05/50 | Train Loss: 0.4859 | Train Acc: 81.34% | Val Loss: 0.6164 | Val Acc: 76.88%
Epoch 06/50 | Train Loss: 0.4265 | Train Acc: 83.62% | Val Loss: 0.3070 | Val Acc: 87.92%
Saved New Best Model (Acc: 87.92%)
Epoch 07/50 | Train Loss: 0.4014 | Train Acc: 84.78% | Val Loss: 0.4999 | Val Acc: 78.79%
Epoch 08/50 | Train Loss: 0.3808 | Train Acc: 85.62% | Val Loss: 1.1377 | Val Acc: 63.77%
Epoch 09/50 | Train Loss: 0.3370 | Train Acc: 87.41% | Val Loss: 0.5831 | Val Acc: 80.12%
Epoch 10/50 | Train Loss: 0.3192 | Train Acc: 87.78% | Val Loss: 0.7875 | Val Acc: 77